In [0]:
import importlib
import sys
from pathlib import Path
from pyspark.sql import functions as F
project_root = str(Path.cwd().resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

import utils.storage_config as storage_config
from pyspark.sql import functions as F
importlib.reload(storage_config)
storage_config.spark = spark
storage_config.configure_storage()

In [0]:
df_customers = spark.read.format("delta")\
    .load("abfss://bronze@secondstorage89.dfs.core.windows.net/customers/")


In [0]:
df_customers.show(10)

In [0]:
df_customers.printSchema()

In [0]:
df_customers.count()

In [0]:
df_customers.select("customer_unique_id").distinct().count()

customer_id identifies a specific customer record in the dataset. That's why every row has a unique customer_id. In our dataset, there are 99,441 rows and 99,441 unique customer IDs.

customer_unique_id, on the other hand, identifies the actual customer/person. The same customer can have multiple records/orders, so they can have different customer_id values but the same customer_unique_id.

In [0]:
print("Total rows:", df_customers.count())
print("Unique customer_id:", df_customers.select("customer_id").distinct().count())
print("Unique customer_unique_id:", df_customers.select("customer_unique_id").distinct().count())

In [0]:
df_customers.groupBy("customer_id") \
  .count() \
  .filter("count > 1") \
  .show()

In [0]:
df_customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_customers.columns
]).show()

check if zip code is negative or not

In [0]:
df_customers.select(
    "customer_zip_code_prefix"
).describe().show()

In [0]:
df_customers.select("customer_city") \
    .distinct() \
    .orderBy("customer_city") \
    .show(50, truncate=False)

customer zip code is in int so we convert this into string format

In [0]:


silver_customer = (
    df_customers
    .withColumn(
        "customer_zip_code_prefix",
        F.col("customer_zip_code_prefix").cast("string")
    )
)

In [0]:
silver_customer.printSchema()

In [0]:
silver_customer.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@secondstorage89.dfs.core.windows.net/customers/")